In [ ]:
import os
from pathlib import Path
from string import ascii_lowercase

import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

if "MPL_STYLE" not in os.environ:
    os.environ["MPL_STYLE"] = "seaborn-v0_8-notebook"
plt.style.use(os.environ["MPL_STYLE"])


In [ ]:
OUTPUT_BASE = Path(os.getenv("OUTPUT_BASE")).resolve(strict=True)

SUMMARY_DIR = OUTPUT_BASE / "summary" / "M6"
SUMMARY_CSV_PATH = (SUMMARY_DIR / "summary.csv").resolve(strict=True)
SCORE_CSV_PATH = (SUMMARY_DIR / "score.csv").resolve(strict=True)


In [ ]:
summary_df = pd.read_csv(SUMMARY_CSV_PATH)
summary_df

In [ ]:
print(
    summary_df.columns,
    summary_df["model_family"].unique(),
    summary_df["feature_set"].unique(),
    summary_df["target"].unique(),
    sep="\n",
)

In [ ]:
score_df = pd.read_csv(SCORE_CSV_PATH)
score_df

In [ ]:
print(
    score_df.columns,
)

In [ ]:
MODEL_FAMILY = {
    "summary_stats": {
        "label": "Summary Stats",
        "label_short": "SS",
        "color": "#1f77b4",
        "marker": "o",
    },
    "deep_sets": {
        "label": "Deep Sets",
        "label_short": "DS",
        "color": "#d62728",
        "marker": "s",
    },
    "set_transformer": {
        "label": "Set Transformer",
        "label_short": "ST",
        "color": "#2ca02c",
        "marker": "^",
    },
}

FEATURE_SET_PLOT_CONFIG = {
    "Sky": {"label": "Sky", "color": "#1f77b4"},
    "Sky+L": {"label": "Sky$+L$", "color": "#6baed6"},
    "Cartesian": {"label": "Cartesian", "color": "#d62728"},
    "Cartesian+L": {"label": "Cartesian$+L$", "color": "#fb6a4a"},
}

TARGET_PLOT_CONFIG = {
    "age": {"label": "Age", "unit": "Myr"},
    "total_mass": {"label": "Total Mass", "unit": "$M_\\odot$"},
}

## Visualization

In [ ]:
for target_label, plot_config in TARGET_PLOT_CONFIG.items():
    fig, axs = plt.subplots(1, 4, figsize=(16, 4), dpi=300, gridspec_kw={"wspace": 0})

    target_df = summary_df[summary_df["target"] == target_label].copy()

    for ax_idx, (ax, feature_set_label) in enumerate(zip(axs, FEATURE_SET_PLOT_CONFIG)):
        feature_df = target_df[target_df["feature_set"] == feature_set_label]

        ax.set_title(FEATURE_SET_PLOT_CONFIG[feature_set_label]["label"], fontsize=20)
        ax.text(
            0.95,
            0.95,
            f"({ascii_lowercase[ax_idx]})",
            transform=ax.transAxes,
            va="top",
            ha="right",
            fontsize=20,
        )

        for model_family, family_cfg in MODEL_FAMILY.items():
            model_df = feature_df[feature_df["model_family"] == model_family]

            ax.scatter(
                model_df["snapshot_mean_phys_ae_median"],
                model_df["snapshot_mean_phys_ae_q90"],
                s=100,
                c=family_cfg["color"],
                marker=family_cfg["marker"],
                alpha=0.9,
                label=family_cfg["label"],
                edgecolors="black",
                linewidths=0.5,
            )

        ax_line_cfg = {"ls": ":", "lw": 0.8, "c": "gray", "alpha": 0.8}
        if target_label == "age":
            ax.set_xlim(0, 40)
            ax.xaxis.set_major_locator(mpl.ticker.MultipleLocator(20, offset=10))
            ax.xaxis.set_minor_locator(mpl.ticker.MultipleLocator(5))
            ax.set_ylim(50, 130)
            ax.yaxis.set_major_locator(mpl.ticker.MultipleLocator(20))
            ax.yaxis.set_minor_locator(mpl.ticker.MultipleLocator(10))
            [ax.axhline(y, **ax_line_cfg) for y in [60, 80, 100, 120]]
            [ax.axvline(x, **ax_line_cfg) for x in [10, 30]]
        else:
            ax.set_xlim(0, 18)
            ax.xaxis.set_major_locator(mpl.ticker.MultipleLocator(6, offset=3))
            ax.xaxis.set_minor_locator(mpl.ticker.MultipleLocator(3))
            ax.set_ylim(9, 51)
            ax.yaxis.set_major_locator(mpl.ticker.MultipleLocator(12, offset=3))
            ax.yaxis.set_minor_locator(mpl.ticker.MultipleLocator(6, offset=3))
            [ax.axhline(y, **ax_line_cfg) for y in [15, 27, 39]]
            [ax.axvline(x, **ax_line_cfg) for x in [3, 15]]

        if ax in axs[1:]:
            ax.set_yticklabels([])
        else:
            ax.set_ylabel(
                r"$Q_{90}\;\mathrm{AE}_\mathrm{snap.}$" + f" [{plot_config['unit']}]"
            )

    legend_entries = {
        label: handle
        for legend_ax in axs
        for handle, label in zip(*legend_ax.get_legend_handles_labels())
    }

    fig.supxlabel(
        r"$\mathrm{MedAE}_\mathrm{snap.}$" + f" [{plot_config['unit']}]", y=-0.08
    )
    fig.legend(
        list(legend_entries.values()),
        list(legend_entries.keys()),
        loc="upper center",
        ncol=4,
        markerscale=1.4,
        frameon=True,
        bbox_to_anchor=(0.5, 1.14),
    )

    fig.savefig(SUMMARY_DIR / f"analysis-{target_label}.pdf", bbox_inches="tight")
